# 01 · The forward process — turning data into noise

Our plan is to destroy data with noise and learn to undo it. This notebook builds
the *destroying* half, carefully. By the end you'll have derived the single
equation that makes diffusion models practical.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each is followed by a
> **self-check** cell that verifies your implementation and prints ✅.
>
> **Stuck?** The answer key is `solutions/notebooks/`, and the reference
> implementation lives in the `nanodiffusion/` package. Peeking is allowed —
> but try first.


## 1. One small step of noise

We corrupt the data over $T$ steps. A single step is defined as:

$$q(x_t \mid x_{t-1}) = \mathcal N\!\big(x_t;\ \underbrace{\sqrt{1-\beta_t}}_{\text{shrink}}\,x_{t-1},\ \underbrace{\beta_t I}_{\text{add noise}}\big)$$

In plain words, **two things happen at every step**:

1. **Shrink** the current sample slightly, multiplying by $\sqrt{1-\beta_t}$.
2. **Add** fresh Gaussian noise with variance $\beta_t$.

$\beta_t$ is a small number (like 0.0001 → 0.02) that we choose. The list
$\beta_1,\dots,\beta_T$ is called the **noise schedule**.

### Why shrink? (variance preservation)

The shrinking looks arbitrary, but it's doing something important. Recall that
for a constant $c$, $\operatorname{Var}(c\,x) = c^2\operatorname{Var}(x)$. So if
our data starts with variance 1:

$$\operatorname{Var}(x_t) = \underbrace{(1-\beta_t)\cdot 1}_{\text{from shrinking}} + \underbrace{\beta_t}_{\text{from new noise}} = 1$$

The variance stays exactly **1** forever. Without the shrink, we'd keep piling on
noise and the numbers would blow up. This design is called a
**variance-preserving** (VP) process — the data smoothly *morphs* into standard
Gaussian noise instead of exploding into it.

That word *preserving* is doing real work: there is a rival design where the
variance genuinely **does** explode, and it works perfectly well too. We'll build
it in §7 once you have VP working.

## 2. The reparameterization trick

Writing "$x_t$ is a *sample from* $\mathcal N(\mu, \sigma^2)$" is awkward for
algebra. The **reparameterization trick** rewrites any Gaussian sample as a
deterministic formula plus one standard noise draw:

$$x \sim \mathcal N(\mu, \sigma^2 I) \quad\Longleftrightarrow\quad x = \mu + \sigma\,\varepsilon,\qquad \varepsilon \sim \mathcal N(0, I)$$

All the randomness is isolated into $\varepsilon$. Applying it to our step:

$$\boxed{\;x_t = \sqrt{1-\beta_t}\;x_{t-1} + \sqrt{\beta_t}\;\varepsilon_t\;}$$

Now it's just algebra we can manipulate. (This trick is also what makes the whole
thing differentiable, which matters for training.)

**Notation.** From here on define $\alpha_t = 1-\beta_t$, so the step reads

$$x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t$$

## 3. Deriving the "nice property" ⭐

Here's the problem: to train, we need $x_t$ for a random $t$. Simulating 500
little steps every time would be painfully slow. Can we jump straight from $x_0$
to $x_t$?

Yes — and here's the derivation. We only need one fact from probability:

> **Sum of independent Gaussians:** if $a\sim\mathcal N(0,\sigma_a^2)$ and
> $b\sim\mathcal N(0,\sigma_b^2)$ are independent, then
> $a+b\sim\mathcal N(0,\ \sigma_a^2+\sigma_b^2)$. **Variances add.**

**Step 1 — write two consecutive steps:**

$$x_t = \sqrt{\alpha_t}\,x_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t$$
$$x_{t-1} = \sqrt{\alpha_{t-1}}\,x_{t-2} + \sqrt{1-\alpha_{t-1}}\,\varepsilon_{t-1}$$

**Step 2 — substitute the second into the first:**

$$x_t = \sqrt{\alpha_t}\Big(\sqrt{\alpha_{t-1}}\,x_{t-2} + \sqrt{1-\alpha_{t-1}}\,\varepsilon_{t-1}\Big) + \sqrt{1-\alpha_t}\,\varepsilon_t$$

$$x_t = \sqrt{\alpha_t\alpha_{t-1}}\;x_{t-2} \;+\; \underbrace{\sqrt{\alpha_t(1-\alpha_{t-1})}\,\varepsilon_{t-1} + \sqrt{1-\alpha_t}\,\varepsilon_t}_{\text{two independent Gaussians}}$$

**Step 3 — merge the two noise terms.** Their variances add:

$$\alpha_t(1-\alpha_{t-1}) + (1-\alpha_t) = \alpha_t - \alpha_t\alpha_{t-1} + 1 - \alpha_t = 1 - \alpha_t\alpha_{t-1}$$

Look at that cancellation — the $\alpha_t$ terms vanish. So the two noises collapse
into a single one:

$$x_t = \sqrt{\alpha_t\alpha_{t-1}}\;x_{t-2} + \sqrt{1-\alpha_t\alpha_{t-1}}\;\varepsilon$$

**Step 4 — spot the pattern.** That has *exactly the same shape* as one step, but
with $\alpha_t\alpha_{t-1}$ in place of $\alpha_t$. Keep unrolling all the way to
$x_0$ and the products accumulate. Defining the running product

$$\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$$

we get the **nice property**:

$$\boxed{\;x_t = \sqrt{\bar\alpha_t}\;x_0 \;+\; \sqrt{1-\bar\alpha_t}\;\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I)\;}$$

**Why this is a big deal:** noising to *any* timestep is now **one line of code**,
$O(1)$ instead of $O(t)$. That's what lets us train on a random $t$ each step.

## 4. Reading $\bar\alpha_t$ as a signal dial

The nice property has a lovely interpretation. $x_t$ is a **weighted blend**:

$$x_t = \underbrace{\sqrt{\bar\alpha_t}}_{\text{how much signal}}\,x_0 + \underbrace{\sqrt{1-\bar\alpha_t}}_{\text{how much noise}}\,\varepsilon$$

and the two weights always satisfy $(\sqrt{\bar\alpha_t})^2+(\sqrt{1-\bar\alpha_t})^2=1$
— the variance-preservation from §1, now visible at every $t$.

| $\bar\alpha_t$ | meaning |
|---|---|
| $\approx 1$ (small $t$) | almost pure data, a whisper of noise |
| $\approx 0.5$ | half signal, half noise — the interesting middle |
| $\approx 0$ (large $t$) | signal erased; $x_T$ is indistinguishable from $\mathcal N(0,I)$ |

That last row is what lets us **start generation from pure noise**: if
$\bar\alpha_T\approx 0$, then noise is a valid starting point.

A useful summary number is the **signal-to-noise ratio**
$\mathrm{SNR}(t)=\dfrac{\bar\alpha_t}{1-\bar\alpha_t}$, which falls monotonically
from huge to ~0.

In [ ]:
import torch
import matplotlib.pyplot as plt

from nanodiffusion.utils import set_seed, scatter_2d
from nanodiffusion.data import toy2d
# reference implementations, used ONLY by the self-check cells:
from nanodiffusion.schedules import NoiseSchedule, linear_beta_schedule, cosine_beta_schedule
from nanodiffusion.forward import add_noise as reference_add_noise

set_seed(0)
T = 200
data = toy2d("swiss_roll", 8000)
print("data:", tuple(data.shape))

## TODO 1 — compute $\bar\alpha_t$ from the betas

Turn the schedule into the running product. Given `betas` of length $T$:

- $\alpha_t = 1-\beta_t$
- $\bar\alpha_t = \prod_{s\le t}\alpha_s$ — a **cumulative product**

*Hint:* `torch.cumprod(x, dim=0)` returns `[x0, x0*x1, x0*x1*x2, ...]`.

In [ ]:
def my_alpha_bars(betas: torch.Tensor) -> torch.Tensor:
    '''Return alpha_bar_t = prod_{s<=t} (1 - beta_s), same length as betas.'''
    alpha_t = 1 - betas
    return torch.cumprod(alpha_t, dim=0)

In [ ]:
# ---- self-check 1 ----
betas = linear_beta_schedule(T)
mine = my_alpha_bars(betas)
ref = NoiseSchedule(betas).alpha_bars
assert mine.shape == ref.shape, f"shape {mine.shape} != {ref.shape}"
assert torch.allclose(mine, ref, atol=1e-6), "values don't match the reference"
assert torch.all(mine[:-1] >= mine[1:]), "alpha_bar must be non-increasing"
print(f"✅ TODO 1 correct — signal decays from {mine[0]:.3f} to {mine[-1]:.4f}")
print("   (that's the *linear* schedule at T=200; §5 explains why it doesn't reach 0)")

## 5. Choosing the schedule: linear vs cosine

We get to pick the $\beta_t$. Two popular choices:

- **Linear** (original DDPM, 2020): $\beta_t$ ramps linearly from $10^{-4}$ to
  $0.02$ — a range tuned for $T=1000$ steps.
- **Cosine** (Improved DDPM, 2021): define $\bar\alpha_t$ *directly* from a cosine
  curve of the **fraction** $t/T$, then read the betas back out.

That difference — absolute $\beta$ range vs. a curve in $t/T$ — matters more than
it looks. Run the cell and compare the two columns.

**At $T=1000$** (what DDPM used) the linear schedule crushes the signal too
early: $\bar\alpha$ is already $0.08$ at the halfway point, and **327 of the 1000
steps** sit below $\bar\alpha<0.01$ — a third of the network's capacity spent on
inputs already indistinguishable from noise. Cosine holds $\bar\alpha\approx0.49$
at the midpoint and wastes only 65. *This* is the classic argument for cosine.

**At $T=200$** (what we use) linear has the **opposite** problem. Its $\beta$
range is hardcoded for 1000 steps, so across only 200 steps it never finishes the
job: $\bar\alpha_T = 0.13$, meaning $x_T$ still carries visible signal. That
quietly **breaks generation** — we start sampling from pure $\mathcal N(0,I)$, but
the forward process never actually got there, so the two ends don't meet.

Because cosine is defined in terms of $t/T$, it **adapts to whatever $T$ you
choose**, reaching $\bar\alpha_T\approx0$ in both columns. That's exactly why the
rest of these notebooks use **cosine with $T=200$** — it keeps cells fast without
breaking the math.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

# beta curves at our T
axes[0].plot(linear_beta_schedule(T), label="linear")
axes[0].plot(cosine_beta_schedule(T), label="cosine")
axes[0].set_title(r"$\beta_t$ (noise per step), $T=200$"); axes[0].set_xlabel("t"); axes[0].legend()

# alpha_bar at T=200 and T=1000, plotted against the FRACTION t/T so they compare
for ax, TT in zip(axes[1:], (200, 1000)):
    lin_ab = my_alpha_bars(linear_beta_schedule(TT))
    cos_ab = my_alpha_bars(cosine_beta_schedule(TT))
    frac = torch.linspace(0, 1, TT)
    ax.plot(frac, lin_ab, label=f"linear (ends {lin_ab[-1]:.3f})")
    ax.plot(frac, cos_ab, label=f"cosine (ends {cos_ab[-1]:.3f})")
    ax.axhline(0, ls=":", c="gray", lw=1)
    ax.set_title(r"$\bar\alpha_t$ (signal surviving), " + f"$T={TT}$")
    ax.set_xlabel("t / T"); ax.set_ylim(-0.05, 1.05); ax.legend()

plt.tight_layout(); plt.show()

for TT in (200, 1000):
    lin = my_alpha_bars(linear_beta_schedule(TT))
    cos = my_alpha_bars(cosine_beta_schedule(TT))
    print(f"T={TT:5d} | linear: mid={lin[TT//2]:.3f} end={lin[-1]:.4f} "
          f"wasted(<0.01)={int((lin < 0.01).sum()):4d}"
          f"   || cosine: mid={cos[TT//2]:.3f} end={cos[-1]:.4f} "
          f"wasted={int((cos < 0.01).sum()):4d}")

# our setting must actually reach pure noise, or sampling from N(0, I) is invalid
assert my_alpha_bars(cosine_beta_schedule(200))[-1] < 0.01
print("\ncosine @ T=200 reaches ~0 ✔  (linear @ T=200 would not)")

## TODO 2 — implement the nice property

Time to write the boxed equation from §3:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon$$

**One practical wrinkle: shapes.** Our data `x0` is `(B, 2)` and `t` is `(B,)` —
a *different* timestep per item. So `alpha_bars[t]` gives `(B,)`, one scalar per
item, and you must reshape it to `(B, 1)` before multiplying, so it broadcasts
across both coordinates of each point.

In [ ]:
def my_add_noise(x0: torch.Tensor, t: torch.Tensor, alpha_bars: torch.Tensor,
                 noise: torch.Tensor | None = None):
    '''Noise x0 to timestep t in one shot.

    Args:
        x0:         (B, 2) clean data
        t:          (B,) long tensor of timesteps
        alpha_bars: (T,) the cumulative products from TODO 1
        noise:      optional (B, 2) eps; sampled from N(0, I) if None
    Returns:
        (x_t, noise)   -- we return the noise too, because training needs
                          the exact eps the network will be asked to predict.
    '''
    if noise is None:
        noise = torch.randn_like(x0)

    x_t = torch.sqrt(alpha_bars[t].reshape(-1, 1)) * x0 + torch.sqrt(1 - alpha_bars[t].reshape(-1, 1)) * noise

    return x_t, noise

In [ ]:
# ---- self-check 2 ----
schedule = NoiseSchedule.make("cosine", T)
ab = my_alpha_bars(cosine_beta_schedule(T))
t = torch.randint(0, T, (data.shape[0],))
eps = torch.randn_like(data)                      # fixed noise, so we can compare

x_mine, _ = my_add_noise(data, t, ab, noise=eps)
x_ref, _ = reference_add_noise(data, t, schedule, noise=eps)
assert x_mine.shape == data.shape, f"shape {x_mine.shape} != {data.shape}"
assert torch.allclose(x_mine, x_ref, atol=1e-5), "doesn't match the reference"

# at t = T-1 the data should be ~pure unit-variance noise (this is what makes it
# legitimate to *start* generation from N(0, I) later)
x_T, _ = my_add_noise(data, torch.full((data.shape[0],), T - 1), ab)
print(f"alpha_bar_T = {ab[-1]:.4f}  (want ~0)")
print(f"std(x_T)    = {x_T.std().item():.4f}  (want ~1)")
assert abs(x_T.std().item() - 1.0) < 0.15
print("✅ TODO 2 correct — the forward process works")

## 6. See it work

Watch the swiss roll dissolve using **your** implementation. Notice it doesn't
drift or explode — it *morphs* into a unit Gaussian blob. That's variance
preservation doing its job.

In [ ]:
ts = [0, 10, 25, 50, 100, 199]
fig, axes = plt.subplots(1, len(ts), figsize=(2.6 * len(ts), 2.6))
for ax, ti in zip(axes, ts):
    x_t, _ = my_add_noise(data, torch.full((data.shape[0],), ti), ab)
    scatter_2d(ax, x_t, f"t = {ti}   " + r"$\bar\alpha$=" + f"{ab[ti]:.2f}")
plt.suptitle("Your forward process: swiss roll -> noise")
plt.tight_layout(); plt.show()

## 7. The road not taken: variance **exploding**

Everything so far followed one design choice — *shrink, then add noise*. There's
a second family, developed independently by Song & Ermon (2019) for score-based
models, which drops the shrinking entirely:

$$\textbf{VE:}\quad x_t = x_0 + \sigma_t\,\varepsilon \qquad\qquad \textbf{VP:}\quad x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$$

Instead of a schedule of $\beta$'s, VE picks a **geometric ladder of noise
scales**, $\sigma_{\min}=0.01$ up to $\sigma_{\max}\approx 50$. The data is never
touched — we just bury it under progressively louder noise. So

$$\operatorname{Var}(x_t)=\operatorname{Var}(x_0)+\sigma_t^2$$

which grows without bound. Hence **variance exploding**.

### Why does that still work?

Because what actually matters isn't that the variance equals 1 — it's that by the
final step **the data is completely swamped**. If $\sigma_{\max}$ is much larger
than the biggest distance between two data points, then $x_T$ tells you
essentially nothing about $x_0$, so starting generation from
$\mathcal N(0,\sigma_{\max}^2I)$ is legitimate.

> **VP shrinks the signal down to nothing. VE drowns it under something enormous.**
> Two routes to the same destination: a final state independent of the data.

### The trade-off

| | VP (DDPM) | VE (SMLD / NCSN) |
|---|---|---|
| formula | $\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$ | $x_0+\sigma_t\varepsilon$ |
| variance | stays 1 | $1+\sigma_t^2\to\infty$ |
| network input scale | always ~unit | spans orders of magnitude |
| start sampling from | $\mathcal N(0,I)$ | $\mathcal N(0,\sigma_{\max}^2I)$ |
| the original $x_0$ | shrunk away | left untouched |

VP's constant scale is much friendlier to neural networks — inputs stay
normalized at every $t$ — which is a big part of why VP-style diffusion dominates
image generation. VE's advantage is conceptual: because $x_0$ is never rescaled,
$x_t$ is *literally the data convolved with a Gaussian*, which is exactly the
setting where the **score function** $\nabla_x\log p(x)$ has a clean form and
Langevin dynamics gives a natural sampler.

### The punchline: they're the same thing

Song et al. (2021) showed both are discretizations of stochastic differential
equations:

$$\textbf{VE:}\ \ dx=\sqrt{\tfrac{d[\sigma^2(t)]}{dt}}\;dw \qquad\qquad \textbf{VP:}\ \ dx=-\tfrac12\beta(t)\,x\,dt+\sqrt{\beta(t)}\;dw$$

Spot the difference: VP has a **drift** term $-\frac12\beta(t)x$ pulling $x$
toward zero — that's the shrinking — while VE is pure diffusion with no drift.
Same family, different member. **Notebook 07** develops this properly.

For the rest of Part 1 we use **VP**, but let's see VE with our own eyes.

In [ ]:
import math

# VE: a geometric ladder of noise scales, instead of a beta schedule
sigma_min, sigma_max = 0.01, 50.0
sigmas = torch.exp(torch.linspace(math.log(sigma_min), math.log(sigma_max), T))

def ve_add_noise(x0, t, sigmas, noise=None):
    '''VE forward process: add noise of scale sigma_t. No shrinking at all.'''
    if noise is None:
        noise = torch.randn_like(x0)
    s = sigmas[t].reshape(-1, 1)
    return x0 + s * noise, noise

ts = [0, 50, 100, 150, 199]
fig, axes = plt.subplots(2, len(ts), figsize=(2.5 * len(ts), 5.4))
for j, ti in enumerate(ts):
    tt = torch.full((data.shape[0],), ti)
    x_vp, _ = my_add_noise(data, tt, ab)
    x_ve, _ = ve_add_noise(data, tt, sigmas)
    scatter_2d(axes[0, j], x_vp, f"VP  t={ti}", lim=2.5)
    scatter_2d(axes[1, j], x_ve, f"VE  t={ti}  " + r"$\sigma$=" + f"{sigmas[ti]:.2f}",
               lim=max(2.5, 3 * sigmas[ti].item()), color="C3")
plt.suptitle("Same data, two forward processes "
             "— note the axis numbers on the bottom row", y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# The variance story, analytically. Our data is normalized to Var(x0) = 1.
vp_var = ab + (1 - ab)          # = 1 at every t, by construction
ve_var = 1.0 + sigmas ** 2

plt.figure(figsize=(6.5, 3.2))
plt.plot(vp_var, label="VP — preserved at 1")
plt.plot(ve_var, label="VE — explodes", color="C3")
plt.yscale("log"); plt.xlabel("t"); plt.ylabel(r"Var($x_t$), log scale")
plt.title("Why they're called what they're called"); plt.legend(); plt.show()

print(f"VP  Var: {vp_var[0]:.2f} -> {vp_var[-1]:.2f}")
print(f"VE  Var: {ve_var[0]:.2f} -> {ve_var[-1]:,.0f}"
      f"   (noise scale sigma_max = {sigmas[-1]:.0f})")

# Both destroy the data, by opposite means: VP kills the signal, VE buries it.
assert abs(vp_var[-1].item() - 1.0) < 1e-4, "VP should preserve variance"
assert ve_var[-1] > 100, "VE should explode"
print("\nBoth reach a data-independent end state ✔")

## 8. Schedule playground — roll your own

Nothing forces us to use linear or cosine. The schedule is a free design choice,
so let's build the machinery to try any of them.

### The recipe

Designing $\beta_t$ directly is treacherous, because
$\bar\alpha_t=\prod_s(1-\beta_s)$ is a **product** — small changes in $\beta$
compound exponentially, and $\bar\alpha$ is what the model actually experiences.
So do it the other way round: **draw the $\bar\alpha$ curve you want, then read
the betas back out**:

$$\beta_t = 1-\frac{\bar\alpha_t}{\bar\alpha_{t-1}}$$

That's exactly how `cosine_beta_schedule` works in `nanodiffusion/schedules.py` —
and it accepts *any* monotone curve from 1 down to 0.

### The rules a schedule must obey

1. $0<\beta_t<1$ — it's a variance, and we need $\alpha_t=1-\beta_t>0$
2. $\bar\alpha_0=1$ — start clean
3. $\bar\alpha_T\approx 0$ — end at pure noise (**this is the one linear@$T{=}200$ breaks**)
4. $\bar\alpha$ monotonically decreasing

Plus a *soft* guideline: keep $\beta_t$ small, since the "reverse step is
Gaussian" argument in notebook 02 assumes small steps.

In [ ]:
import math

def alpha_bars_to_betas(ab_full: torch.Tensor) -> torch.Tensor:
    '''Turn a curve ab_full (length T+1, from 1.0 down to ~0) into T betas.'''
    betas = 1.0 - ab_full[1:] / ab_full[:-1]
    return betas.clamp(1e-8, 0.999)      # clipping matters -- see the table below

def inspect(name, ab_full):
    '''Check the four hard rules and report practical diagnostics.'''
    betas = alpha_bars_to_betas(ab_full)
    hard = {
        "0<beta<1":   bool((betas > 0).all() and (betas < 1).all()),
        "abar_0~1":   abs(ab_full[0].item() - 1.0) < 0.05,
        "abar_T~0":   ab_full[-1].item() < 0.01,
        "monotone":   bool((ab_full[:-1] >= ab_full[1:] - 1e-9).all()),
    }
    dead = int((betas < 1e-6).sum())          # steps that add no noise at all
    return betas, hard, dead

u = torch.linspace(0, 1, T + 1)               # normalized time, t/T

def cosine_c(u, s=0.008):
    f = torch.cos((u + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0]).clamp(min=1e-8)
def exponential_c(u, c=8.0):  return torch.exp(-c * u).clamp(min=1e-8)
def sigmoid_c(u, k=10.0):
    r = torch.sigmoid(-k * (u - 0.5))
    return ((r - r[-1]) / (r[0] - r[-1])).clamp(min=1e-8)
def sqrt_c(u, s=1e-4):        return (1 - torch.sqrt(u + s)).clamp(min=1e-8)
def relu_c(u, u0=0.3):        return torch.clamp(1 - (u - u0) / (1 - u0), 1e-8, 1.0)
def linear_c(u):
    b = linear_beta_schedule(len(u) - 1)
    return torch.cat([torch.ones(1), torch.cumprod(1 - b, 0)])

SCHEDULES = {"linear": linear_c, "cosine": cosine_c, "exponential": exponential_c,
             "sigmoid": sigmoid_c, "sqrt": sqrt_c, "relu": relu_c}

print(f"{'schedule':12s} {'abar_T':>9s} {'max beta':>9s} {'last beta':>10s} "
      f"{'dead':>5s}  rules")
print("-" * 74)
for name, fn in SCHEDULES.items():
    ab_full = fn(u)
    betas, hard, dead = inspect(name, ab_full)
    failed = [k for k, v in hard.items() if not v]
    verdict = "OK" if not failed else "FAILS: " + ",".join(failed)
    print(f"{name:12s} {ab_full[-1]:9.2e} {betas[:-1].max():9.4f} "
          f"{betas[-1]:10.4f} {dead:5d}  {verdict}")

### Reading that table

**`linear` is the only outright failure** — $\bar\alpha_T=0.13$, so it never
reaches noise at $T{=}200$ (our §5 finding, now caught automatically).

**`exponential` is the best-behaved**, and there's a neat reason. If
$\bar\alpha_t=e^{-cu}$ then $\alpha_t=e^{-c/T}$ is *constant*, so $\beta$ is
constant too — 0.039 at every single step. **An exponential $\bar\alpha$ is
exactly a constant-$\beta$ schedule.** It reaches $\bar\alpha_T\approx3\times10^{-4}$
with uniformly tiny steps.

**The big `last beta` values (0.999) are not bugs.** Any curve that drives
$\bar\alpha$ to *exactly* zero at $t=T$ must have $\alpha_T\to0$, i.e.
$\beta_T\to1$. That's why every real implementation — including
`cosine_beta_schedule` — **clips $\beta$ at 0.999**. Improved-DDPM does precisely
this. It's numerically delicate but fine: at that noise level the sampler's first
reverse step relies on a cancellation, which is why our sampler works.

**`relu` shows the cost of a flat region.** With $\bar\alpha=1$ for the first 30%
of the schedule, those steps add *literally no noise* — see the `dead` column.
They're wasted compute: 60 timesteps where the network learns nothing.

**`sqrt` starts at 0.99, not 1.0.** That's deliberate in Diffusion-LM: the offset
$s$ means even $t=0$ carries a whisper of noise, avoiding a degenerate first step.
Our `abar_0~1` rule allows 5% slack for exactly this reason — a schedule may
legitimately begin slightly below 1.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
for name, fn in SCHEDULES.items():
    ab_full = fn(u)
    style = dict(lw=2.4, zorder=3) if name == "cosine" else dict(lw=1.4, alpha=0.85)
    a1.plot(u, ab_full, label=name, **style)
    ab = ab_full.clamp(1e-7, 1 - 1e-7)
    a2.plot(u, torch.log(ab / (1 - ab)), label=name, **style)

a1.set_title(r"$\bar\alpha_t$ — the signal dial"); a1.set_xlabel("t / T")
a1.axhline(0, ls=":", c="gray", lw=1); a1.legend(fontsize=8)

a2.set_title(r"log-SNR  $\log\frac{\bar\alpha}{1-\bar\alpha}$ — what the model experiences")
a2.set_xlabel("t / T"); a2.axhline(0, ls=":", c="gray", lw=1); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

The **log-SNR** panel on the right is the more meaningful view: it's the curve the
model actually experiences, and where it's flat is where the schedule spends most
of its timesteps.

This connects to a deep result — Kingma et al.'s *Variational Diffusion Models*
(2021) showed that in continuous time the ELBO depends **only on the endpoints**
of the log-SNR curve, not its shape in between. The shape controls *which noise
levels get training emphasis* (and the variance of the gradient estimator), not
the optimum itself. So schedule design is less "find the true curve" and more
**"decide where to spend your capacity."**

Now the same thing on actual data — the swiss roll at the **same timestep** under
different schedules:

In [ ]:
show = ["exponential", "cosine", "sqrt", "relu"]
ts_show = [40, 100, 160]
fig, axes = plt.subplots(len(show), len(ts_show), figsize=(2.5 * len(ts_show), 2.5 * len(show)))
for i, name in enumerate(show):
    ab_sched = SCHEDULES[name](u)[1:]            # per-step alpha_bars, length T
    for j, ti in enumerate(ts_show):
        x_t, _ = my_add_noise(data, torch.full((data.shape[0],), ti), ab_sched)
        scatter_2d(axes[i, j], x_t,
                   f"{name}  t={ti}  " + r"$\bar\alpha$=" + f"{ab_sched[ti]:.2f}")
plt.suptitle("Same t, same data — the schedule decides how much is left", y=1.0)
plt.tight_layout(); plt.show()

### 🔧 Your turn

Edit the curve below and re-run. Some things worth trying:

- a **power law** `(1 - u) ** p` — how does $p$ change where noise concentrates?
- a **piecewise** curve that lingers at high SNR then drops fast
- push `relu`'s `u0` to 0.6 and watch the `dead` count climb
- change `exponential`'s `c` — how small can it get before `abar_T~0` fails?

In [ ]:
def my_schedule(u):
    '''Your alpha_bar curve. Must start at 1.0 and decrease to ~0.  <- EDIT ME'''
    p = 3.0
    return torch.clamp((1 - u) ** p, min=1e-8)

ab_full = my_schedule(u)
betas, hard, dead = inspect("mine", ab_full)
print("hard rules:", {k: ("OK" if v else "FAIL") for k, v in hard.items()})
print(f"abar_T = {ab_full[-1]:.2e} | max beta (pre-terminal) = {betas[:-1].max():.4f} "
      f"| dead steps = {dead}")

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
axes[0].plot(u, ab_full); axes[0].plot(u, cosine_c(u), ls="--", c="gray", label="cosine")
axes[0].set_title(r"your $\bar\alpha$"); axes[0].set_xlabel("t / T"); axes[0].legend(fontsize=8)
for ax, ti in zip(axes[1:], [40, 100, 160]):
    x_t, _ = my_add_noise(data, torch.full((data.shape[0],), ti), ab_full[1:])
    scatter_2d(ax, x_t, f"t={ti}  " + r"$\bar\alpha$=" + f"{ab_full[1:][ti]:.2f}", color="C2")
plt.tight_layout(); plt.show()

if all(hard.values()):
    print("\n✅ valid schedule — you could train with this one")
else:
    print("\n⚠️  breaks a hard rule; see which above")

## Recap

- A step **shrinks then adds noise**, keeping variance at 1 — **variance
  preserving**.
- The **reparameterization trick** turns sampling into algebra.
- Because **variances of independent Gaussians add**, $t$ steps collapse into one:
  $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$ — the **nice property**.
- $\bar\alpha_t$ is the signal dial; at $t=T$ it's ~0, so $x_T$ is just noise.
- The **cosine** schedule adapts to any $T$; a linear one tuned for $T{=}1000$
  quietly fails at $T{=}200$.
- **VE** is the alternative forward process: no shrinking, exploding variance.
  It reaches the same data-independent end state by drowning the signal instead of
  shrinking it — and it's the same SDE family in disguise (notebook 07).
- Schedules are a **free design choice**: draw any monotone $\bar\alpha$ curve and
  derive the betas. What matters is the four hard rules — above all
  $\bar\alpha_T\approx0$ — and *where on the log-SNR axis you spend your steps*.

**The open question for notebook 02:** given a noisy $x_t$, how do we step
*backwards*? (Spoiler: it comes down to guessing the $\varepsilon$ in that boxed
equation.)